# Notebook 03 — Streaming Simulado
## NYC Taxi Trips — Big Data Pipeline

**Objetivo:** Simular procesamiento incremental (streaming) de viajes de taxi usando micro-batches por fecha.

**Estrategia:**
- Dividir el dataset procesado en micro-batches diarios
- Procesar cada batch simulando llegada de datos en tiempo real
- Acumular métricas por batch
- Guardar resultados en capa curated

In [1]:
# ─── Celda 1: Imports y rutas ─────────────────────────────────────────────────
import os
import time
import pandas as pd
import numpy as np
import psycopg2
from psycopg2.extras import execute_values
from datetime import datetime, timedelta

BASE_PATH      = os.path.abspath("../")
PROCESSED_PATH = os.path.join(BASE_PATH, "data", "processed")
CURATED_PATH   = os.path.join(BASE_PATH, "data", "curated")
SCREENS_PATH   = os.path.join(BASE_PATH, "screenshots")
STREAMING_PATH = os.path.join(BASE_PATH, "data", "streaming_batches")

os.makedirs(CURATED_PATH,   exist_ok=True)
os.makedirs(STREAMING_PATH, exist_ok=True)

print("Rutas configuradas ✅")
print("CURATED_PATH  :", CURATED_PATH)
print("STREAMING_PATH:", STREAMING_PATH)

Rutas configuradas ✅
CURATED_PATH  : c:\Users\IV4M\Desktop\IA\8vo\DatosMasivos\nyc_taxi_project\data\curated
STREAMING_PATH: c:\Users\IV4M\Desktop\IA\8vo\DatosMasivos\nyc_taxi_project\data\streaming_batches


In [2]:
# ─── Celda 2: Cargar datos procesados ────────────────────────────────────────
print("Cargando datos procesados...")

processed_file = os.path.join(PROCESSED_PATH, "trips_processed.parquet")
pdf = pd.read_parquet(processed_file)

# Convertir fechas
pdf["pickup_dt"] = pd.to_datetime(pdf["tpep_pickup_datetime"])
pdf["pickup_date"] = pdf["pickup_dt"].dt.date

print(f"Filas cargadas : {len(pdf):,}")
print(f"Rango de fechas: {pdf['pickup_date'].min()} → {pdf['pickup_date'].max()}")
print(f"Días únicos    : {pdf['pickup_date'].nunique()}")
print("Carga completada ✅")

Cargando datos procesados...
Filas cargadas : 8,780,868
Rango de fechas: 2002-12-31 → 2023-04-05
Días únicos    : 100
Carga completada ✅


In [3]:
# ─── Celda 3: Preparar micro-batches por día ──────────────────────────────────
print("Preparando micro-batches...")

# Tomar solo enero para la simulación (más rápido)
pdf_jan = pdf[pdf["pickup_month"] == 1].copy()
dates   = sorted(pdf_jan["pickup_date"].unique())

# Crear archivos de batch por día
batch_files = []
for date in dates:
    day_data  = pdf_jan[pdf_jan["pickup_date"] == date]
    file_path = os.path.join(STREAMING_PATH, f"batch_{date}.parquet")
    day_data.to_parquet(file_path, index=False)
    batch_files.append((date, file_path, len(day_data)))

print(f"Micro-batches creados: {len(batch_files)}")
print(f"\nPrimeros 5 batches:")
for date, path, rows in batch_files[:5]:
    print(f"  {date}  →  {rows:,} viajes")
print("\nMicro-batches preparados ✅")

Preparando micro-batches...
Micro-batches creados: 33

Primeros 5 batches:
  2003-01-01  →  2 viajes
  2009-01-01  →  1 viajes
  2023-01-01  →  69,646 viajes
  2023-01-02  →  61,635 viajes
  2023-01-03  →  80,248 viajes

Micro-batches preparados ✅


In [4]:
# ─── Celda 4: Crear tabla streaming en PostgreSQL ─────────────────────────────
conn = psycopg2.connect(
    host="localhost", port=5432,
    database="nyc_taxi_db", user="postgres", password=""
)
cur = conn.cursor()

cur.execute("""
DROP TABLE IF EXISTS streaming_daily_metrics CASCADE;
CREATE TABLE streaming_daily_metrics (
    batch_id           SERIAL PRIMARY KEY,
    batch_date         DATE,
    processed_at       TIMESTAMP,
    total_trips        INT,
    total_revenue      FLOAT,
    avg_fare           FLOAT,
    avg_distance       FLOAT,
    avg_duration_min   FLOAT,
    avg_speed_mph      FLOAT,
    cash_trips         INT,
    credit_trips       INT,
    weekend_trips      INT
);
""")

conn.commit()
print("Tabla streaming_daily_metrics creada ✅")
cur.close()
conn.close()

Tabla streaming_daily_metrics creada ✅


In [5]:
# ─── Celda 5: Simulación de streaming ────────────────────────────────────────
print("Iniciando simulación de streaming...")
print("=" * 55)

accumulated_metrics = []
total_processed     = 0

conn = psycopg2.connect(
    host="localhost", port=5432,
    database="nyc_taxi_db", user="postgres", password=""
)
cur = conn.cursor()

for i, (date, file_path, n_rows) in enumerate(batch_files):
    batch_start = time.time()

    # Leer micro-batch
    batch_df = pd.read_parquet(file_path)

    # Calcular métricas del batch
    metrics = {
        "batch_date"       : str(date),
        "processed_at"     : datetime.now().isoformat(),
        "total_trips"      : len(batch_df),
        "total_revenue"    : round(batch_df["total_amount"].sum(), 2),
        "avg_fare"         : round(batch_df["fare_amount"].mean(), 2),
        "avg_distance"     : round(batch_df["trip_distance"].mean(), 2),
        "avg_duration_min" : round(batch_df["trip_duration_min"].mean(), 2),
        "avg_speed_mph"    : round(batch_df["avg_speed_mph"].mean(), 2),
        "cash_trips"       : int((batch_df["payment_type_name"] == "Cash").sum()),
        "credit_trips"     : int((batch_df["payment_type_name"] == "Credit Card").sum()),
        "weekend_trips"    : int(batch_df["is_weekend"].sum())
    }

    # Insertar en PostgreSQL
    cur.execute("""
        INSERT INTO streaming_daily_metrics
        (batch_date, processed_at, total_trips, total_revenue, avg_fare,
         avg_distance, avg_duration_min, avg_speed_mph, cash_trips, credit_trips, weekend_trips)
        VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
    """, (
        metrics["batch_date"], metrics["processed_at"],
        metrics["total_trips"], metrics["total_revenue"],
        metrics["avg_fare"], metrics["avg_distance"],
        metrics["avg_duration_min"], metrics["avg_speed_mph"],
        metrics["cash_trips"], metrics["credit_trips"],
        metrics["weekend_trips"]
    ))
    conn.commit()

    total_processed += metrics["total_trips"]
    elapsed = round(time.time() - batch_start, 2)
    accumulated_metrics.append(metrics)

    print(f"Batch {i+1:02d} | {date} | {metrics['total_trips']:>6,} viajes | "
          f"Revenue: ${metrics['total_revenue']:>10,.0f} | {elapsed}s")

    # Simular delay de streaming (0.1s entre batches)
    time.sleep(0.1)

cur.close()
conn.close()

print("=" * 55)
print(f"\nTotal batches procesados : {len(batch_files)}")
print(f"Total viajes procesados  : {total_processed:,}")
print("\n✅ Simulación de streaming completada")

Iniciando simulación de streaming...
Batch 01 | 2003-01-01 |      2 viajes | Revenue: $       191 | 0.05s
Batch 02 | 2009-01-01 |      1 viajes | Revenue: $        16 | 0.01s
Batch 03 | 2023-01-01 | 69,646 viajes | Revenue: $ 2,140,335 | 0.13s
Batch 04 | 2023-01-02 | 61,635 viajes | Revenue: $ 1,934,760 | 0.12s
Batch 05 | 2023-01-03 | 80,248 viajes | Revenue: $ 2,356,681 | 0.05s
Batch 06 | 2023-01-04 | 89,296 viajes | Revenue: $ 2,513,117 | 0.08s
Batch 07 | 2023-01-05 | 94,921 viajes | Revenue: $ 2,594,736 | 0.06s
Batch 08 | 2023-01-06 | 96,383 viajes | Revenue: $ 2,573,241 | 0.06s
Batch 09 | 2023-01-07 | 98,872 viajes | Revenue: $ 2,535,655 | 0.06s
Batch 10 | 2023-01-08 | 79,863 viajes | Revenue: $ 2,281,003 | 0.05s
Batch 11 | 2023-01-09 | 79,849 viajes | Revenue: $ 2,246,699 | 0.05s
Batch 12 | 2023-01-10 | 94,095 viajes | Revenue: $ 2,502,532 | 0.06s
Batch 13 | 2023-01-11 | 99,942 viajes | Revenue: $ 2,633,969 | 0.06s
Batch 14 | 2023-01-12 | 104,179 viajes | Revenue: $ 2,883,758 | 0.

In [6]:
# ─── Celda 6: Resumen de streaming y capa curated ─────────────────────────────
df_metrics = pd.DataFrame(accumulated_metrics)

print("=== Resumen de streaming ===")
print(f"Días procesados   : {len(df_metrics)}")
print(f"Total viajes      : {df_metrics['total_trips'].sum():,}")
print(f"Revenue total     : ${df_metrics['total_revenue'].sum():,.2f}")
print(f"Fare promedio     : ${df_metrics['avg_fare'].mean():.2f}")
print(f"Distancia promedio: {df_metrics['avg_distance'].mean():.2f} mi")
print(f"Duración promedio : {df_metrics['avg_duration_min'].mean():.1f} min")

# Guardar métricas en capa curated
curated_file = os.path.join(CURATED_PATH, "streaming_metrics.parquet")
df_metrics.to_parquet(curated_file, index=False)
print(f"\nMétricas guardadas en: {curated_file}")

# También guardar CSV para evidencia
csv_file = os.path.join(SCREENS_PATH, "streaming_metrics_evidence.csv")
df_metrics.to_csv(csv_file, index=False)
print(f"Evidencia CSV en    : {csv_file}")

print("\n✅ Notebook 03 completado exitosamente.")

=== Resumen de streaming ===
Días procesados   : 33
Total viajes      : 2,875,615
Revenue total     : $78,518,593.89
Fare promedio     : $20.53
Distancia promedio: 3.85 mi
Duración promedio : 15.2 min

Métricas guardadas en: c:\Users\IV4M\Desktop\IA\8vo\DatosMasivos\nyc_taxi_project\data\curated\streaming_metrics.parquet
Evidencia CSV en    : c:\Users\IV4M\Desktop\IA\8vo\DatosMasivos\nyc_taxi_project\screenshots\streaming_metrics_evidence.csv

✅ Notebook 03 completado exitosamente.
